# Pipeline mestre — Censo 2022 e análise territorial da RMR

Notebook orquestrador. A lógica analítica permanece em módulos versionados em `src/censo_rmr/`; este notebook controla ambiente, configuração, contratos, execução, QA e proveniência.

## 0. Montagem do Google Drive e instalação do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/censo_senso_rmr
!git clone -b agent/pipeline-rmr-v0 https://github.com/hilaliskandar/censo_senso_rmr.git /content/censo_senso_rmr
%cd /content/censo_senso_rmr
!pip -q install -e .


## 1. Configuração central

In [ ]:
from pathlib import Path
from censo_rmr.configuracao import carregar_configuracao
from censo_rmr.contratos import carregar_produtos, verificar_produto

REPO = Path('/content/censo_senso_rmr')
CFG = carregar_configuracao(REPO / 'config/config.yaml')
PRODUTOS = carregar_produtos(REPO / 'config/produtos.yaml')
DRIVE_RAIZ = Path('/content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR')
assert DRIVE_RAIZ.exists(), f'Pasta do Drive não encontrada: {DRIVE_RAIZ}'
print('Municípios:', len(CFG.municipios))


## 2. Pré-voo

Valida estrutura de pastas, presença dos produtos históricos e disponibilidade dos módulos antes de qualquer reprocessamento.

In [ ]:
pastas = CFG.dados['drive']['pastas']
for nome, relativo in pastas.items():
    caminho = DRIVE_RAIZ / relativo
    print(f'{nome:28s}', 'OK' if caminho.exists() else 'AUSENTE', caminho)


In [ ]:
for nome in ['demografia', 'composicao_domestica', 'renda', 'cruzamentos_habitacionais']:
    v = verificar_produto(DRIVE_RAIZ, PRODUTOS, nome)
    print(f'{nome:28s}', 'CONTRATO HISTÓRICO OK' if v.ok else f'AUSENTES: {list(v.ausentes)}')


## 3. Módulos já reconstruídos

Os módulos abaixo possuem lógica separada de I/O e testes automatizados. A produção de novos arquivos permanece desativada até a validação de regressão integral contra os produtos históricos do Drive.

In [ ]:
from censo_rmr.demografia import preparar_demografia_setorial, resumir_demografia_municipios
from censo_rmr.composicao_domestica import *  # interface ainda em consolidação
from censo_rmr.renda import preparar_renda_setorial, resumir_renda_municipios
from censo_rmr.cruzamentos import calcular_limiares_cruzamentos, cruzar_renda_composicao

STATUS_MODULOS = {
    'demografia': 'reconstruído + regressão unitária histórica',
    'composicao_domestica': 'refatorado + testes unitários',
    'renda': 'refatorado + testes unitários',
    'cruzamentos': 'refatorado; limiares deixaram de ser hardcoded',
    'fcu': 'pendente',
    'entorno': 'pendente',
    'vulnerabilidade': 'pendente',
    'lisa': 'produto histórico localizado; código não localizado',
    'segregacao': 'produto histórico localizado; código não localizado',
    'pca_tipologias': 'produto histórico localizado; código não localizado',
}
STATUS_MODULOS


## 4. Etapas do pipeline

1. aquisição/cache IBGE;
2. inspeção de dicionários e esquema;
3. recorte territorial RMR;
4. demografia;
5. composição doméstica;
6. rendimento;
7. cruzamentos habitacionais;
8. FCU e equidade;
9. entorno e condições habitacionais;
10. vulnerabilidade e densidade;
11. Moran/LISA e sensibilidade;
12. segregação;
13. PCA e tipologias;
14. tabelas e mapas;
15. manifesto e relatório de QA.

In [ ]:
ETAPAS_ATIVAS = ['pre_voo', 'auditoria_contratos']
ETAPAS_EM_REGRESSAO = ['demografia', 'composicao_domestica', 'renda', 'cruzamentos']
print('Execução automática habilitada:', ETAPAS_ATIVAS)
print('Módulos aguardando regressão integral:', ETAPAS_EM_REGRESSAO)


## 5. Manifesto de execução

In [ ]:
from censo_rmr.proveniencia import manifesto_execucao, salvar_manifesto

manifesto = manifesto_execucao(
    configuracao=CFG.dados,
    avisos=[
        'Pipeline v0: pré-voo e contratos habilitados.',
        'Blocos 1 a 3 reconstruídos/refatorados, ainda aguardando regressão integral antes de sobrescrever produtos históricos.'
    ],
)
saida = DRIVE_RAIZ / '08_Relatorio_Consolidado' / 'manifestos' / 'MANIFESTO_EXECUCAO_V0.json'
salvar_manifesto(manifesto, saida)
print(saida)
